# 00_common
Shared Bronze→Silver loader: typing, quarantine, dedupe, MERGE and a per-table watermark. Other notebooks load it with `%run 00_common`.

In [ ]:
from pyspark.sql import functions as F, Window

spark.sql("""CREATE TABLE IF NOT EXISTS etl_watermark
             (table_name STRING, last_batch_id STRING) USING DELTA""")

def get_wm(t):
    r = spark.sql(f"SELECT last_batch_id FROM etl_watermark WHERE table_name = '{t}'").collect()
    return r[0][0] if r else ""

def set_wm(t, b):
    spark.sql(f"""
        MERGE INTO etl_watermark w
        USING (SELECT '{t}' AS table_name, '{b}' AS last_batch_id) s
        ON w.table_name = s.table_name
        WHEN MATCHED THEN UPDATE SET last_batch_id = s.last_batch_id
        WHEN NOT MATCHED THEN INSERT *""")

def load_silver(src, tgt, keys, cols_sql, rules=None):
    """Bronze -> Silver: type, quarantine, dedupe on business key, MERGE, move watermark."""
    wm = get_wm(tgt)

    # 1. only batches newer than the watermark (watermark = arrival batch, not business date)
    df = spark.sql(f"SELECT {cols_sql}, _batch_id, _ingested_at FROM {src} WHERE _batch_id > '{wm}'")
    if df.limit(1).count() == 0:
        print(f"{tgt:30} nothing new (watermark {wm})")
        return
    new_wm = df.agg(F.max("_batch_id")).first()[0]

    # 2. quarantine rows that break a rule (never silently drop)
    if rules:
        reason = F.lit(None).cast("string")
        for cond, msg in rules:
            reason = F.when(F.expr(cond), F.lit(msg)).otherwise(reason)
        df = df.withColumn("_dq_reason", reason)
        bad = df.filter("_dq_reason IS NOT NULL")
        if spark.catalog.tableExists("silver_quarantine"):   # safe rerun: no duplicate quarantine rows
            spark.sql(f"DELETE FROM silver_quarantine WHERE target_table = '{tgt}' AND _batch_id > '{wm}'")
        (bad.select(F.lit(tgt).alias("target_table"),
                    F.to_json(F.struct(*bad.columns)).alias("row_json"),
                    F.col("_dq_reason").alias("reason"),
                    "_batch_id",
                    F.current_timestamp().alias("quarantined_at"))
            .write.format("delta").mode("append").saveAsTable("silver_quarantine"))
        df = df.filter("_dq_reason IS NULL").drop("_dq_reason")

    # 3. dedupe: keep the latest version of each business key
    w = Window.partitionBy(*keys).orderBy(F.col("SystemModifiedAt").desc(), F.col("_ingested_at").desc())
    df = df.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")

    # 4. MERGE -> idempotent
    if not spark.catalog.tableExists(tgt):
        df.write.format("delta").saveAsTable(tgt)
    else:
        df.createOrReplaceTempView("src")
        on = " AND ".join(f"t.{k} = s.{k}" for k in keys)
        spark.sql(f"""
            MERGE INTO {tgt} t USING src s ON {on}
            WHEN MATCHED AND s.SystemModifiedAt >= t.SystemModifiedAt THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *""")

    # 5. move the watermark only after success
    set_wm(tgt, new_wm)
    print(f"{tgt:30} loaded up to {new_wm} -> {spark.table(tgt).count()} rows")